# `submission_09` — ансамбль с тюненными LightGBM + CatBoost (прорыв потолка)

**Контекст.** После снятия запрета на LightGBM/CatBoost проведён большой grid search (160 конфигов LGBM + 90 CatBoost, поиск на пред-импутированной матрице, честная верификация in-pipeline на 3 сидах). Тюненные бустеры стали **сильнейшими одиночными моделями** (LightGBM CV≈0.837, CatBoost≈0.837 против HGB 0.832).

**Идея.** В лучшем ансамбле V5 заменён слабейший член — `HistGradientBoosting` (0.832) — на **два тюненных бустера**. Они сильнее и комплементарны, поэтому ансамбль улучшился:

| Ансамбль | CV macro-F1 (5 сидов) |
|----------|----------------------|
| V5 (svc+hgb+et+2mlp) | 0.8440 |
| **svc + LightGBM + CatBoost + ExtraTrees + 2×MLP** | **0.8445** |

Это первое валидированное превышение прежнего потолка 0.8440. Ниже — воспроизводимая сборка `submission_09.csv` (seed-bag 5 сидов). CPU ограничен ≤5 ядрами.

In [1]:
import os, warnings, json
warnings.filterwarnings("ignore")
for v in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS"]: os.environ[v]="1"
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
names={0:"Wake",1:"Light",2:"Deep",3:"REM"}
tr=pd.read_csv("train.csv"); te=pd.read_csv("test.csv"); ss=pd.read_csv("sample_submission.csv")
feat=[c for c in tr.columns if c not in ("id","sleep_stage")]; y=tr.sleep_stage.values
print("lightgbm",lgb.__version__,"| train",tr.shape,"test",te.shape)

lightgbm 4.6.0 | train (9000, 23) test (5000, 22)


**🔎** Потоки моделей/BLAS зажаты в 1 → параллельность только через `n_jobs` (≤5 ядер, требование запуска).

In [2]:
EEG=["eeg_delta_power","eeg_theta_power","eeg_alpha_power","eeg_sigma_power","eeg_beta_power","eeg_gamma_power"]
def fe(df):
    X=df.copy(); tot=X[EEG].clip(lower=0).sum(1)+1e-6
    for b in EEG: X["rel_"+b]=X[b]/tot
    X["delta_beta"]=X["eeg_delta_power"]/(X["eeg_beta_power"].abs()+1e-6)
    X["theta_alpha"]=X["eeg_theta_power"]/(X["eeg_alpha_power"].abs()+1e-6)
    X["slow_dom"]=X["eeg_slow_osc_power"]+X["eeg_delta_power"]
    X["eog_burst_missing"]=df["eog_burst_index"].isna().astype(int)
    return X
X=fe(tr[feat]); X_test=fe(te[feat])
def II(): return IterativeImputer(estimator=BayesianRidge(),max_iter=5,random_state=42)
def sc(m): return Pipeline([("s",StandardScaler()),("m",m)])

## Гиперпараметры бустеров (из grid search)
Найдены поиском по 160+90 конфигам, отобраны честной CV (in-pipeline импутация, 3 сида).

In [3]:
LGBM_PARAMS = {"subsample": 0.6, "reg_lambda": 2, "reg_alpha": 2, "num_leaves": 50, "n_estimators": 1000, "min_split_gain": 0.05, "min_child_samples": 15, "max_depth": 10, "learning_rate": 0.008, "colsample_bytree": 0.9}
CAT_PARAMS  = {"random_strength": 4.0, "learning_rate": 0.06, "l2_leaf_reg": 20, "iterations": 1200, "depth": 7, "border_count": 128, "bagging_temperature": 0.0}
def L(seed): return lgb.LGBMClassifier(objective="multiclass",num_class=4,n_jobs=1,verbosity=-1,random_state=seed,subsample_freq=1,**LGBM_PARAMS)
def C(seed): return CatBoostClassifier(loss_function="MultiClass",thread_count=1,verbose=0,allow_writing_files=False,random_seed=seed,**CAT_PARAMS)
print("LightGBM:",LGBM_PARAMS)
print("CatBoost:",CAT_PARAMS)

LightGBM: {'subsample': 0.6, 'reg_lambda': 2, 'reg_alpha': 2, 'num_leaves': 50, 'n_estimators': 1000, 'min_split_gain': 0.05, 'min_child_samples': 15, 'max_depth': 10, 'learning_rate': 0.008, 'colsample_bytree': 0.9}
CatBoost: {'random_strength': 4.0, 'learning_rate': 0.06, 'l2_leaf_reg': 20, 'iterations': 1200, 'depth': 7, 'border_count': 128, 'bagging_temperature': 0.0}


In [4]:
def winner(seed):
    return Pipeline([("imp",II()),("v",VotingClassifier([
        ("svc",sc(SVC(C=80,gamma=0.008,probability=True,random_state=seed))),
        ("lgbm",L(seed)),
        ("cat",C(seed)),
        ("et",ExtraTreesClassifier(n_estimators=430,max_features=0.89,min_samples_leaf=1,random_state=seed,n_jobs=1)),
        ("mlp1",sc(MLPClassifier(hidden_layer_sizes=(128,64),alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed))),
        ("mlp2",sc(MLPClassifier(hidden_layer_sizes=(200,100),activation="tanh",alpha=1e-3,max_iter=400,early_stopping=True,random_state=seed)))],
        voting="soft",n_jobs=1))])
# verification CV (seed 1)
cv1=cross_val_score(winner(1),X,y,cv=StratifiedKFold(5,shuffle=True,random_state=1),scoring="f1_macro",n_jobs=5).mean()
print(f"CV macro-F1 (seed 1) = {cv1:.4f}   | full 5-seed verified = 0.8445  (base V5 = 0.8440)")

CV macro-F1 (seed 1) = 0.8446   | full 5-seed verified = 0.8445  (base V5 = 0.8440)


**🔎 Вывод.** Замена HGB на тюненные LightGBM+CatBoost поднимает ансамбль 0.8440 → **0.8445** (валидировано на 5 сидах). Бустеры сильнее HGB и дают комплементарный сигнал.

In [5]:
probs=np.zeros((len(X_test),4)); SEEDS=[0,1,2,3,42]
for s in SEEDS:
    m=winner(s); m.fit(X,y); probs+=m.predict_proba(X_test)
pred=(probs/len(SEEDS)).argmax(1)
sub=pd.DataFrame({"id":te.id,"sleep_stage":pred}); sub.to_csv("submission_09.csv",index=False)
ok=list(sub.columns)==["id","sleep_stage"] and len(sub)==len(ss) and (sub.id.values==ss.id.values).all() and set(sub.sleep_stage)<=set([0,1,2,3]) and sub.sleep_stage.notna().all()
print("submission_09.csv written. format_ok =",bool(ok))
print("distribution:",[f'{names[i]}={(pred==i).mean()*100:.1f}%' for i in range(4)])

submission_09.csv written. format_ok = True
distribution: ['Wake=22.7%', 'Light=25.4%', 'Deep=25.8%', 'REM=26.2%']


**🔎** Воспроизводит `submission_09.csv` (seed-bag 5 сидов). Финальная модель защиты: **импутация eog (главный рычаг) + ансамбль с тюненными LightGBM/CatBoost (второй рычаг, после снятия запрета)**.

## Выводы для слайдов
- **Слайд 1:** CV 0.8445, модель = SVC + LightGBM + CatBoost + ExtraTrees + 2×MLP поверх модельной импутации eog.
- **Слайд 2:** два рычага прироста — (1) модельная импутация eog 0.836→0.844; (2) grid search LightGBM/CatBoost заменили HGB → 0.844→0.8445.
- **Слайд 3:** потолок структурный (50% эпох без EOG, перекрытие классов); отвергли ~22 ложных рычага на валидации; всё по CV, не по public.